## Bucket Massvis Fixation

Created on 20210318

@author: Yao Wang
@usage: bucket fixations with durations and element bounding boxes

In [5]:
# -*- coding: utf-8 -*-
import numpy as np
import os, sys
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy import ndimage
from eval import calculate_metrics
from natsort import natsorted
from glob import glob
import pandas as pd
from tqdm import trange
from skimage.draw import polygon
from polygon_util import bucketColors
sys.path.append("..") 
from preprocess_utils import get_gt_elements, get_accum_bucket
np.set_printoptions(suppress=True)

# change base path to the directory where you have downloaded the salicon data
bp = "/netpool/homes/wangyo/Dataset/Massvis/targets393"
curdir = '../../eyetracking/csv_files/fixationsByVis'
eledir = '../../dataset/csv_files/targets393_elementLabels/elementLabels/'
origimdir = os.path.join(bp, 'targets')
curfiles = natsorted(glob(curdir + '/*'))

mds_bucket = np.zeros((14, 20))
dur_bucket = np.zeros((14, 600))

### Stats for fixation numbers and durations

In [7]:
outpath = os.path.join(bp, 'plot_overall')
os.makedirs(outpath, exist_ok=True)

mds_bucket = np.zeros((14, 20))


for id in trange(0,len(curfiles), 1):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementCoords, elementX, elementY, boxes = get_gt_elements(imname)


    # load the raw image
    with Image.open(os.path.join(origimdir, imname+'.png')) as im:
        width, height = im.size # original image size

        #Handle Overlap problem
        id_map = np.full([width,height], 13)

        for tt in range(12, -1, -1):
            #print(tt)
            for box in boxes:
                if box.id == tt:
                    rr, cc = polygon(np.array(box.coords)[:,0], np.array(box.coords)[:,1], (width, height))
                    id_map[rr, cc] = tt
        
        allfiles = glob(os.path.join(curfiles[id], 'enc' ,'*.csv'))

        #Add the image to the path
        fig = plt.figure(figsize=(16,16))

        #plt.imshow(im)

        for subcsv in allfiles:
            fixations = pd.read_csv(subcsv, header=None, names=['index','x','y','time'])

            x = fixations['x'].tolist()
            y = fixations['y'].tolist()
            duration = fixations['time'].tolist()

            accum_time = 0
            for row2 in fixations.iterrows():
                accum_time += row2[1][3]
                if (int(row2[0]) < 2): continue
                # If the fixation is inside the box, add one to the bucket counter

                id1 = id_map[int(row2[1][1]),int(row2[1][2])]
                id2 = get_accum_bucket(accum_time)   
                mds_bucket[id1][id2] += 1
                if(row2[1][3]<600):
                    dur_bucket[id1][int(row2[1][3])] += 1

            # expell the first fixation
            #x.pop(0)
            #y.pop(0)
            #duration.pop(0)

            #plt.plot(x,y, '-ro', color='yellow',lw=1)
            #visualize the plot on image
            '''
            for i in range(len(x)):
                #if(x[i] > width): print('error')
                #if(y[i] > height): print('error')
                dur = duration[i]/10 if duration[i]/10>10 else 10
                dur = dur if dur<20 else 20
                if(i == 0):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.5) ) # 
                elif(i == len(x) - 1):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.2), mec='#e9b96e')
                else:
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.5), mec='#e9b96e') 
                plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=5)
            
        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColors(boxes[i].id))

        plt.title(imname+'_total')
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()
        '''
print(mds_bucket)
np.savez('dur.npz',dur_bucket)

100%|██████████| 393/393 [01:59<00:00,  3.28it/s]


[[  15.  302.  354.  419.  463.  509.  572.  629.  607.  614.  599.  643.
   652.  615.  591.  535.  494.  323.  157.   26.]
 [  44.  788. 1052. 1168. 1396. 1501. 1630. 1707. 1819. 1788. 1735. 1824.
  1708. 1666. 1577. 1311. 1114.  758.  295.   46.]
 [  25.  382.  486.  416.  337.  297.  279.  244.  202.  181.  173.  163.
   186.  155.  130.  136.  117.   98.   45.   12.]
 [  18.  490.  788.  933.  952. 1033. 1055. 1072. 1031. 1051. 1037.  977.
   920.  873.  797.  708.  579.  408.  167.   25.]
 [  11.  279.  290.  279.  257.  258.  254.  269.  274.  284.  288.  304.
   294.  281.  260.  234.  216.  171.  102.   27.]
 [  42.  501.  461.  521.  585.  644.  680.  684.  698.  728.  735.  692.
   722.  715.  699.  630.  576.  448.  245.   64.]
 [ 197. 3233. 3759. 2615. 1957. 1467. 1142.  891.  803.  712.  689.  562.
   504.  481.  510.  452.  369.  211.   71.    9.]
 [  13.  253.  286.  299.  313.  298.  283.  260.  288.  258.  242.  215.
   209.  209.  158.  177.  142.   83.   43.    4.]


<Figure size 1152x1152 with 0 Axes>

<Figure size 1152x1152 with 0 Axes>

<Figure size 1152x1152 with 0 Axes>

...

<Figure size 1152x1152 with 0 Axes>

<Figure size 1152x1152 with 0 Axes>

<Figure size 1152x1152 with 0 Axes>

<Figure size 1152x1152 with 0 Axes>

<Figure size 1152x1152 with 0 Axes>

### Plot fixation distributions

In [8]:
res = mds_bucket.copy()

for i in range(np.shape(res)[1]):
    res[:,i] /= np.sum(res[:,i])
    res[:,i] = np.round(100*res[:,i], 2)

res

array([[ 3.55,  3.43,  4.01,  4.75,  4.3 ,  3.57,  2.54,  2.72,  2.82,
         3.67,  4.35,  4.63,  3.99,  4.63,  3.58,  3.24,  3.61,  4.55,
         4.57,  4.88],
       [ 4.64,  8.09,  8.7 ,  9.83, 10.26,  8.57,  8.33,  9.37,  6.9 ,
         7.95,  6.67,  6.17,  7.71,  7.72,  7.16,  6.47,  6.94,  5.97,
         6.1 ,  6.59],
       [ 1.64,  2.94,  1.67,  3.05,  2.65,  2.86,  1.45,  2.11,  1.88,
         3.36,  3.19,  4.01,  2.93,  1.23,  1.65,  1.47,  1.67,  1.7 ,
         1.52,  1.22],
       [ 1.91,  3.19,  4.68,  5.08,  3.64,  2.14,  1.09,  0.91,  0.63,
         0.61,  0.87,  0.62,  0.53,  0.93,  0.55,  0.88,  0.56,  0.85,
         0.91,  0.73],
       [ 5.74,  8.82,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.31,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ],
       [ 2.19,  2.45,  2.34,  2.03,  2.98,  1.79,  4.71,  1.51,  3.13,
         3.06,  2.32,  2.47,  2.13,  3.09,  3.31,  3.24,  2.5 ,  3.12,
         3.35,  2.2 ],
       [ 2

## Stats for fixation numbers and durations on Predictions

In [7]:
METHOD_NAME='PathGAN_finetuned'
#METHOD_NAME='PathGAN_unfinetuned'

mds_bucket = np.zeros((14, 20))

outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
predpath = os.path.join(bp, METHOD_NAME, 'scanpaths')
os.makedirs(outpath, exist_ok=True)

SUBJECTS = 25
imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)



    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,imname+'.png.csv'))
        df_pred.columns = ['user', 'x', 'y', 'time']

        #Add the image to the path
        fig = plt.figure(figsize=(16,16))
        plt.imshow(im)

        #for pp in range(SUBJECTS):
        for pp in range(1):
            df_predTmp = df_pred[df_pred['user'] == pp]
            df_predI = df_predTmp.drop(['user'] ,axis=1)
    
            x = df_predI['x'].tolist()
            y = df_predI['y'].tolist()
            duration = df_predI['time'].tolist()

            accum_time = 0
            for row2 in df_predI.iterrows():
                accum_time += row2[1][2]
                if (int(row[0]) < 1): continue
                for box in boxes:
                    if isPoiWithinPoly([int(row2[1][0]),int(row2[1][1])], box.coords):
                        id2 = get_accum_bucket(accum_time)
                        mds_bucket[box.id][id2] += 1

            # expell the first fixation
            #x.pop(0)
            #y.pop(0)
            #duration.pop(0)

            plt.plot(x,y, '-ro', color='yellow',lw=2)
            #visualize the plot on image
            for i in range(len(x)):
                dur = duration[i]/10 if duration[i]/10>10 else 10
                dur = dur if dur<20 else 20
                if(i == 0):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
                elif(i == len(x) - 1):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
                else:
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
                plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColors(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:21<00:00,  3.12it/s]


In [5]:
METHOD_NAME='nc2020'

mds_bucket = np.zeros((14, 20))

outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
predpath = os.path.join(bp, METHOD_NAME, 'tsv')
os.makedirs(outpath, exist_ok=True)

imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)


    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,'ncnc_'+imname+'.tsv'), sep='\t')
        df_pred.columns = ['x', 'y', 'time']

        #Add the image to the path
        fig = plt.figure(figsize=(16,16))

        plt.imshow(im)

        x = df_pred['x'].tolist()
        y = df_pred['y'].tolist()
        duration = df_pred['time'].tolist()

        accum_time = 0
        for row2 in df_pred.iterrows():
            accum_time += row2[1][2]
            if (int(row[0]) < 1): continue
            for box in boxes:
                if isPoiWithinPoly([int(row2[1][0]),int(row2[1][1])], box.coords):
                    id2 = get_accum_bucket(accum_time)
                    mds_bucket[box.id][id2] += 1
                    
        # expell the last (-1,-1,-1) fixation
        x.pop(-1)
        y.pop(-1)
        duration.pop(-1)

        plt.plot(x,y, '-ro', color='yellow',lw=2)
        #visualize the plot on image
        for i in range(len(x)):
            dur = duration[i]/10 if duration[i]/10>10 else 10
            dur = dur if dur<20 else 20

            if(i == 0):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
            elif(i == len(x) - 1):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
            else:
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
            plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColors(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:16<00:00,  3.89it/s]


In [8]:
METHOD_NAME='PathGAN_finetuned_10'

predpath = os.path.join(bp, METHOD_NAME, 'tsv')
outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
os.makedirs(outpath, exist_ok=True)

imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)


    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,imname+'.png.tsv'), header=None, sep='\t')
        df_pred.columns = ['x', 'y', 'time']

        #Add the image to the path
        fig = plt.figure(figsize=(16,16))

        plt.imshow(im)

        x = df_pred['x'].tolist()
        y = df_pred['y'].tolist()
        duration = df_pred['time'].tolist()

        accum_time = 0
        for row2 in df_pred.iterrows():
            accum_time += row2[1][2]
            if (int(row[0]) < 1): continue
            for box in boxes:
                if isPoiWithinPoly([int(row2[1][0]),int(row2[1][1])], box.coords):
                    id2 = get_accum_bucket(accum_time)
                    mds_bucket[box.id][id2] += 1
                    
        # expell the last (-1,-1,-1) fixation
        x.pop(-1)
        y.pop(-1)
        duration.pop(-1)

        plt.plot(x,y, '-ro', color='yellow',lw=2)
        #visualize the plot on image
        for i in range(len(x)):
            dur = duration[i]/10 if duration[i]/10>10 else 10
            dur = dur if dur<20 else 20

            if(i == 0):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
            elif(i == len(x) - 1):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
            else:
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
            plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColors(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:17<00:00,  3.79it/s]


In [3]:
METHOD_NAME='Saltinet'

mds_bucket = np.zeros((14, 20))

predpath = os.path.join(bp, METHOD_NAME, 'scanpaths')
outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
os.makedirs(outpath, exist_ok=True)

SUBJECTS = 25
imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)



    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,imname+'.png.csv'))
        df_pred.columns = ['user', 'index', 'time', 'x', 'y']
        #Add the image to the path
        fig = plt.figure(figsize=(16,16))
        plt.imshow(im)

        #for pp in range(SUBJECTS):
        for pp in range(2):
            df_predTmp = df_pred[df_pred['user'] == pp + 1]
            df_predI = df_predTmp.drop(['user','index'] ,axis=1)

            df_predI['time'] *= 1000
            x = df_predI['x'].tolist()
            y = df_predI['y'].tolist()
            duration = df_predI['time'].tolist()

            accum_time = 0
            for row2 in df_predI.iterrows():
                accum_time += row2[1][0]
                if (int(row[0]) < 1): continue
                for box in boxes:
                    if isPoiWithinPoly([int(row2[1][1]),int(row2[1][2])], box.coords):
                        id2 = get_accum_bucket(accum_time)
                        mds_bucket[box.id][id2] += 1

            # expell the first fixation
            #x.pop(0)
            #y.pop(0)
            #duration.pop(0)

            plt.plot(x,y, '-ro', color='yellow',lw=2)
            #visualize the plot on image
            for i in range(len(x)):
                dur = duration[i]/10 if duration[i]/10>10 else 10
                dur = dur if dur<20 else 20
                if(i == 0):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
                elif(i == len(x) - 1):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
                else:
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
                plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColors(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:24<00:00,  2.71it/s]
